### O cellref todos os sites descartando apenas "SITUACAO_ERB= DESATIVADO". Sites previstos entram para que o EUTranCell (mandatório) possa obter as informações do site UNLOCKED ainda que seu status ainda esteja desatualizado (PREVISTO) no science

In [ ]:
# Falta fazer a automatizacao da latitude e Longitude

# nao esquecer de fz o replace de AMBOS no siteType

import pandas as pd
from pathlib import Path

lte_dump_dir = Path('C:/cellref/LTE_dump')
files = sorted(lte_dump_dir.glob('*.xlsx'))
science = pd.concat(
    (pd.read_excel(f, header=2) for f in files),
    ignore_index=True
)

science['COUNTRY']= 'BRAZIL'
science['MARKET']= 'VIVO'
science['SiteID']= science['SIG_ERB']+'_'+science['SIG_ESTADO']
science['Cell_Name']= science['SIG_LOGICA']

science['REGION'] = science['SIG_ESTADO'].replace(['AM', 'AP','MA', 'PA', 'RR', 'BA', 'SE', 'MG', 'SP'], ['N', 'N', 'N', 'N', 'N', 'BASE', 'BASE', 'MG', 'SP'])

science['COBERTURA_SETOR'] = science['COBERTURA_SETOR'].replace(['AMBOS', 'OUTDOOR', 'INDOOR'],['Outdoor', 'Outdoor',
                                                                                                          'Indoor'])

science= science[science['SITUACAO_ERB']!= 'DESATIVADO']

science.rename(columns={'NOME_MUNICIPIO': 'CLUSTER', 'AREA_REGISTRO': 'CN','NOM_ERB': 'Site_Name', 'NUM_ENB_ID': 'eNodeB_ID',
                        'SIG_LOGICA': 'Sector', 'NUM_AZIMUTE_ANTENA': 'Azimuth', 'NUM_TILT_MECAN_ANTENA': 'MechanicalTilt',
                       'NUM_TILT_ELETRICO': 'ElectricalTilt', 'DES_MODELO_ANTENA': 'AntennaType', 'FORNECEDOR': 'Vendor',
                       'COBERTURA_SETOR': 'SiteType', 'NUM_ALT_MAX_SUP_ANTENA': 'Height'
                       , 'STR_LATITUDE_SITE': 'Latitude', 'STR_LONGITUDE_SITE': 'Longitude', 'DES_ENDERECO_SITE': 'ENDERECO'
                       ,'NOM_BAIRRO': 'BAIRRO'},inplace= True)


science1= science.loc[:,['COUNTRY', 'REGION','CN','MARKET','CLUSTER','BAIRRO','ENDERECO','SiteID','Site_Name','eNodeB_ID','Longitude',
                   'Latitude','Sector','Cell_Name', 'Azimuth', 'MechanicalTilt', 'ElectricalTilt', 
                   'AntennaType', 'Vendor', 'SiteType', 'Height']]


# Funcao para converter Latitude/Longitude

def convert_coordinates(value):
    try:
        if value is None:
            return None
        s = str(value).strip()
        if s == '' or s.lower() == 'nan':
            return None

        # Formato com separador: -DD-MM-SS ou DD-MM-SS
        if s.count('-') >= 2:
            sign = -1 if s.startswith('-') else 1
            s = s.lstrip('-')
            parts = s.split('-')
            degrees = int(parts[0])
            minutes = int(float(parts[1]))
            seconds = float(parts[2].replace(',', '.'))
        else:
            # Formato compacto com hemisferio, ex: 06S431820 ou 038W382480
            import re

            m = re.match(r'^(\d{2,3})([NSEW])(\d{2})(\d{2})(\d{2})$', s.upper())
            if not m:
                return None
            degrees = int(m.group(1))
            hemisphere = m.group(2)
            minutes = int(m.group(3))
            seconds = int(m.group(4)) + (int(m.group(5)) / 100.0)
            sign = -1 if hemisphere in ('S', 'W') else 1

        # Normaliza segundos/minutos caso exceda
        if seconds >= 60:
            minutes += int(seconds // 60)
            seconds = seconds % 60
        if minutes >= 60:
            degrees += int(minutes // 60)
            minutes = minutes % 60

        decimal = sign * (degrees + minutes / 60 + seconds / 3600)
        return decimal
    except Exception as e:
        print(f"Erro ao processar valor: {value}. Detalhes: {e}")
        return None


# Aplicando a funcao para converter Latitude e Longitude

science1['Latitude'] = science1['Latitude'].apply(convert_coordinates)
science1['Longitude'] = science1['Longitude'].apply(convert_coordinates)


#science1.set_index('COUNTRY', inplace=True)

# Arredonda e converte para Int64 (aceita NaN)
science1['MechanicalTilt'] = pd.to_numeric(
    science1['MechanicalTilt'].astype(str).str.replace(',', '.'), 
    errors='coerce'
).round().astype('Int64')

science1['ElectricalTilt'] = pd.to_numeric(
    science1['ElectricalTilt'].astype(str).str.replace(',', '.'), 
    errors='coerce'
).round().astype('Int64')  # o 'I' maiúsculo trata N/A como N/A evitando erros

science1['Height'] = pd.to_numeric(
    science1['Height'].astype(str).str.replace(',', '.'), 
    errors='coerce'
).round().astype('Int64') 

science1.shape

In [ ]:
#science1['Height']= science1['Height'].replace([0,1,2,3,4,5], [6,6,6,6,6,6])
"""
science1[science1['Height']<6]=0
science1[science1['Azimuth']==360]=0
science1[science1['MechanicalTilt']==360]=0
"""
science1['Height'] = pd.to_numeric(science1['Height'], errors='coerce')
science1['Height'] = science1['Height'].apply(lambda x: 6 if pd.isna(x) else (x if x > 6 else 6))
science1['Azimuth'] = science1['Azimuth'].apply(lambda x: 0 if pd.isna(x) else (x if x != 360 else 0))
science1['MechanicalTilt'] = science1['MechanicalTilt'].apply(lambda x: 0 if pd.isna(x) else (x if x != 360 else 0))
#science1['MechanicalTilt'].value_counts()
science1['Aux']= science1['SiteID'].str[1:] + '_' + science1['Cell_Name']
science1.sample(5)

In [ ]:
eutrancell= pd.read_csv('C:/Camilo/EUtranCell_TDD_FDD.txt', sep='\t',encoding='ISO-8859-1',low_memory=False)

lista = (['AM', 'AP','MA', 'PA', 'RR', 'BA', 'SE', 'MG', 'SP']) # lista de sites para serem analisados

# comando para pegar os parametros de uma lista de eNBs

eutrancell=eutrancell.loc[(eutrancell["UF"].isin(lista)),:]

eutrancell['ECI']= eutrancell['eNBId']*256+eutrancell['cellId']

eutrancell['MCC']= eutrancell['CGI'].str[:3]

eutrancell['MNC']= eutrancell['CGI'].str[3:5]

# 'J' entrou junto com o 'Z': as celulas J todas tem earfcndl 9310 (banda 28) = 700 MHz.
eutrancell['Band']= eutrancell['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'J', 'U', 'P', 'Y', 'C', 'L', 'O'],
                                                        ['2600','2600', '1800', '700', '700', '2100', '2600', '850', '1800', '2300', '2300'])

eutrancell.rename(columns={'earfcndl': 'DL_EARFCN', 'tac': 'TrackingAreaCode'}, inplace=True)

eutrancell1= eutrancell.loc[:, ['UF', 'eNB', 'eNBId','CELL', 'DL_EARFCN', 'TrackingAreaCode','PCI', 'MCC','MNC',
                                'ECI', 'Band','administrativeState']]

eutrancell1['Aux']= eutrancell1['eNB'].str[1:] + '_' + eutrancell1['CELL']

eutrancell1.head()

In [ ]:
# declarar left_on e right_on
df1= pd.merge(eutrancell1,science1, left_on=['Aux'], right_on=['Aux'],how='left')
df1['Beamwidth']= 65
df1['SiteID']= df1['SiteID'].fillna(df1['eNB'])
df1['Sector']= df1['Sector'].fillna(df1['CELL'])
df1['Cell_Name']= df1['Cell_Name'].fillna(df1['CELL'])
df1['eNodeB_ID']= df1['eNodeB_ID'].fillna(df1['eNBId'])
df1.drop(columns='eNBId', inplace=True)

df2= df1.loc[:,['COUNTRY', 'REGION','MARKET','CN','CLUSTER','BAIRRO','ENDERECO','SiteID','Site_Name','eNodeB_ID','Longitude',
                   'Latitude','Sector','Cell_Name', 'ECI', 'DL_EARFCN', 'PCI', 'MCC', 'MNC','TrackingAreaCode',
                'Azimuth', 'Beamwidth', 'Height','MechanicalTilt', 'ElectricalTilt', 
                   'AntennaType', 'Vendor', 'Band','SiteType', 'administrativeState', 'UF','eNB', 'CELL']]
df2.head(2)

In [ ]:
power= pd.read_csv('C:/Vivo Reports/LTE/LTE MOs/SectorCarrier.csv', usecols=['eNB', 'CELL', 'maximumTransmissionPower'],low_memory=False)

power['EPRE']= power['maximumTransmissionPower']/10

power.rename(columns={'eNB': 'eNB_Pwr', 'CELL': 'CELL_Pwr'}, inplace=True)

df3= pd.merge(df2,power, left_on=['SiteID', 'Sector'], right_on=['eNB_Pwr', 'CELL_Pwr'],how='left')

df3['CableLossDL']= 0

df4= df3.loc[:,['COUNTRY', 'REGION','MARKET','CN','CLUSTER','BAIRRO','ENDERECO','SiteID','Site_Name','eNodeB_ID','Longitude',
                   'Latitude','Sector','Cell_Name', 'ECI', 'DL_EARFCN', 'PCI', 'MCC', 'MNC','TrackingAreaCode',
                'Azimuth', 'Beamwidth', 'Height','MechanicalTilt', 'ElectricalTilt', 
                   'AntennaType', 'Vendor', 'Band','SiteType', 'EPRE','CableLossDL','administrativeState', 'UF','eNB', 'CELL']]
df4.head(2)

In [ ]:
# Remove somente duplicidades exatas da chave operacional
df4.drop_duplicates(subset=["eNB", "CELL", "SiteID", "Sector"], inplace=True)

from datetime import datetime
import numpy as np


def first_mode(series):
    """Primeiro valor cadastral valido; preserva NaN se nao houver dado."""
    valid = series.dropna()
    if valid.empty:
        return np.nan
    return valid.iloc[0]


def circular_azimuth_mode(series):
    """Moda do azimute; em empate escolhe o candidato de menor dispersao circular."""
    valid = pd.to_numeric(series, errors="coerce").dropna() % 360
    if valid.empty:
        return np.nan

    counts = valid.value_counts()
    candidates = counts[counts == counts.max()].index.to_numpy(dtype=float)
    if len(candidates) == 1:
        return float(candidates[0])

    values = valid.to_numpy(dtype=float)

    def circular_dispersion(candidate):
        distances = np.abs((values - candidate + 180) % 360 - 180)
        return float(distances.sum())

    return float(min(candidates, key=lambda value: (circular_dispersion(value), value)))


def fill_from_group_mode(frame, group_columns, value_columns):
    """Preenche somente vazios usando uma tabela modal compacta por grupo."""
    suffix = '_GroupReference'
    reference = (
        frame.groupby(group_columns, dropna=False)[value_columns]
        .first()
        .add_suffix(suffix)
        .reset_index()
    )
    frame = frame.merge(reference, how='left', on=group_columns, sort=False)
    for column in value_columns:
        frame[column] = frame[column].fillna(frame[f'{column}{suffix}'])
    return frame.drop(columns=[f'{column}{suffix}' for column in value_columns])


output_date = datetime.now().strftime('%m%d%Y')

# 1) Fotografia fiel do cadastro: usada para apontar inconsistencias a operadora.
original_df = df4.copy()
original_path = f'C:/cellref/LTE/CellRef_LTE_{output_date}.xlsx'
original_df.to_excel(original_path, index=False)

# 2) Cellref operacional do Tool Box: recebe apenas os tratamentos abaixo.
custom_df = original_df.copy()

# O merge original usa o eNB hospedeiro. Em Hotel de BTS isso perde o cadastro do
# site remoto. A referencia exata UF + Cell_Name recupera a celula diretamente na
# Science, sem herdar endereco/coordenada do eNB hospedeiro.
science_reference = science1.copy()
science_reference['_UF_REF'] = science_reference['SiteID'].astype('string').str.rsplit('_', n=1).str[-1]

reference_columns = [
    'COUNTRY', 'REGION', 'CN', 'MARKET', 'CLUSTER', 'BAIRRO', 'ENDERECO',
    'Site_Name', 'Longitude', 'Latitude', 'Azimuth', 'Height',
    'MechanicalTilt', 'ElectricalTilt', 'AntennaType', 'Vendor', 'SiteType'
]

exact_reference = (
    science_reference
    .groupby(['_UF_REF', 'Cell_Name'], dropna=False)[reference_columns]
    .first()
    .reset_index()
)

custom_df = custom_df.merge(
    exact_reference,
    how='left',
    left_on=['UF', 'CELL'],
    right_on=['_UF_REF', 'Cell_Name'],
    suffixes=('', '_Science')
)

for column in reference_columns:
    custom_df[column] = custom_df[column].fillna(custom_df[f'{column}_Science'])

custom_df.drop(
    columns=['_UF_REF', 'Cell_Name_Science'] + [f'{column}_Science' for column in reference_columns],
    inplace=True,
    errors='ignore'
)

# Site logico vem do nome da celula. Isso preserva os sites remotos dos Hoteis de BTS.
custom_df['_LogicalSite'] = custom_df['UF'].astype('string') + '|' + custom_df['CELL'].astype('string').str[1:4]
custom_df['_SectorKey'] = custom_df['CELL'].astype('string').str[-1]
# Setor normalizado para 1/2/3: o ultimo digito do nome pode passar de 3 quando ha
# mais de uma portadora no mesmo setor fisico (4/5/6 = 2a portadora, 7/8/9 = 3a...).
# 4->1, 5->2, 6->3, 7->1, 8->2, 9->3. Usado somente na moda do azimute abaixo.
_ultimo_digito = pd.to_numeric(custom_df['CELL'].astype('string').str[-1], errors='coerce')
custom_df['_SectorKeyNorm'] = ((_ultimo_digito - 1) % 3 + 1)
custom_df['_HostSite'] = custom_df['UF'].astype('string') + '|' + custom_df['eNB'].astype('string').str[1:4]
custom_df['_IsHotel'] = custom_df['_LogicalSite'].ne(custom_df['_HostSite'])

site_columns = [
    'COUNTRY', 'REGION', 'CN', 'MARKET', 'CLUSTER', 'BAIRRO', 'ENDERECO',
    'Site_Name', 'Longitude', 'Latitude', 'Vendor', 'SiteType'
]
sector_fill_columns = ['Height', 'MechanicalTilt', 'ElectricalTilt', 'AntennaType']

# Valores existentes sao preservados; somente os vazios sao completados.
custom_df = fill_from_group_mode(custom_df, ['_LogicalSite'], site_columns)
custom_df = fill_from_group_mode(custom_df, ['_LogicalSite', '_SectorKey'], sector_fill_columns)

# Somente o azimute e corrigido mesmo quando ja possui valor: todas as portadoras
# do mesmo site logico + setor recebem o valor mais frequente (moda) daquele setor.
# O setor usado aqui e o normalizado 1/2/3 (_SectorKeyNorm), para que co-setores de
# portadoras diferentes (sufixos 4/5/6, 7/8/9) caiam no mesmo grupo antes da moda.
# Hoteis de BTS ficam fora desta regra: neles o azimute individual cadastrado e
# preservado, pois setores remotos do mesmo eNB hospedeiro podem ter direcoes distintas.
modal_azimuth = (
    custom_df.loc[~custom_df['_IsHotel']]
    .groupby(['_LogicalSite', '_SectorKeyNorm'], dropna=False)['Azimuth']
    .agg(circular_azimuth_mode)
    .rename('_ModalAzimuth')
    .reset_index()
)
custom_df = custom_df.merge(
    modal_azimuth, how='left', on=['_LogicalSite', '_SectorKeyNorm'], sort=False
)
normal_sector = ~custom_df['_IsHotel']
custom_df.loc[normal_sector, 'Azimuth'] = custom_df.loc[
    normal_sector, '_ModalAzimuth'
].fillna(custom_df.loc[normal_sector, 'Azimuth'])
custom_df.drop(columns=['_ModalAzimuth'], inplace=True)

custom_df.drop(columns=['_LogicalSite', '_SectorKey', '_SectorKeyNorm', '_HostSite', '_IsHotel'], inplace=True)
custom_df = custom_df[original_df.columns]

custom_path = 'C:/cellref/LTE/cellref.xlsx'
custom_df.to_excel(custom_path, index=False)

print(f'Original: {original_path} ({len(original_df):,} linhas)')
print(f'Tool Box: {custom_path} ({len(custom_df):,} linhas)')
